# Stabilité du CNN classique et du CNN avec CBAM
Réglages fixés : Adam, LR 0,001, Dropout 0,5, maximum 20 époques et patience 4 sur le F1 macro de validation.

Les sauvegardes de la configuration 2 (graine 42) sont réutilisées. Quatre nouveaux entraînements sont lancés avec les graines 123 et 2026. Exécuter les cellules dans l'ordre avec le noyau **memoire_cnn**. La dernière cellule lance les calculs. Ne pas lancer deux copies simultanément.

Après une interruption, réexécuter les cellules : chaque entraînement reprend à la dernière époque complète. Les essais terminés sont reconnus. Les fichiers des recherches précédentes ne sont pas modifiés.

Cette analyse décrit la variabilité liée à l'entraînement sur une répartition fixe. Trois graines donnent une estimation limitée. La graine 42 a servi à choisir les hyperparamètres : ses résultats peuvent être favorisés par cette sélection. Les deux nouvelles graines sont également résumées séparément. Aucun test n'est chargé et aucune meilleure graine n'est sélectionnée.


In [ ]:
from pathlib import Path
import json, hashlib, time, shutil, sys
import numpy as np
import pandas as pd
import PIL
from PIL import Image, ImageOps
import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader
from sklearn import __version__ as version_sklearn
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix
from tqdm.auto import tqdm
from IPython.display import display

PROJET = Path(r"C:\Users\David KUNYIMA\OneDrive\Desktop\Memoire_CNN_Attention")
IMAGES = PROJET / "archive" / "chest_xray"
REPARTITION = PROJET / "repartitions" / "repartition_20260917_102959_713272"
RACINE = PROJET / "optimisation" / "stabilite_multigraines_v1"
GRAINE, LOT, MAX_EPOQUES, PATIENCE = 42, 16, 20, 4
APPAREIL = torch.device("cpu")
GRILLE = [(0.001, 0.5)]
CLASSES = {"NORMAL": 0, "PNEUMONIA": 1}
torch.use_deterministic_algorithms(True)
print("Matériel :", APPAREIL, "| PyTorch :", torch.__version__)
display(pd.DataFrame(GRILLE, columns=["taux_apprentissage", "dropout"]))


## Données et architectures
Même préparation et mêmes architectures que les recherches précédentes.

In [ ]:
train_df = pd.read_csv(REPARTITION / "train.csv")
val_df = pd.read_csv(REPARTITION / "val.csv")
for nom, df, attendus in [("train", train_df, {"NORMAL": 1079, "PNEUMONIA": 2721}), ("val", val_df, {"NORMAL": 269, "PNEUMONIA": 684})]:
    assert df["classe"].value_counts().to_dict() == attendus, f"Effectifs inattendus : {nom}"
    assert df["chemin_relatif"].is_unique
    assert not df[["groupe", "empreinte", "chemin_relatif"]].isna().any().any()
    manquants = [p for p in df["chemin_relatif"] if not (IMAGES / p).is_file()]
    if manquants:
        raise FileNotFoundError(f"{nom} : {len(manquants)} images absentes, exemple : {manquants[0]}")
for colonne in ["groupe", "empreinte", "chemin_relatif"]:
    assert set(train_df[colonne]).isdisjoint(val_df[colonne]), f"Chevauchement : {colonne}"

class Radiographies(Dataset):
    def __init__(self, df): self.df = df.reset_index(drop=True)
    def __len__(self): return len(self.df)
    def __getitem__(self, i):
        ligne = self.df.iloc[i]
        with Image.open(IMAGES / ligne["chemin_relatif"]) as image:
            image = ImageOps.pad(image.convert("L"), (224, 224), method=Image.Resampling.BILINEAR, color=0, centering=(0.5, 0.5))
            x = torch.from_numpy(np.array(image, dtype=np.float32) / 255.0).unsqueeze(0)
        return x, CLASSES[ligne["classe"]]

TRAIN, VAL = Radiographies(train_df), Radiographies(val_df)
effectifs = train_df["classe"].value_counts()
POIDS = torch.tensor([len(TRAIN)/(2*effectifs[c]) for c in ["NORMAL", "PNEUMONIA"]], dtype=torch.float32)
print("Apprentissage :", len(TRAIN), "| Validation :", len(VAL))
print("Poids NORMAL / PNEUMONIA :", POIDS.tolist())
print("Vérifications réussies. Aucun ensemble de test chargé.")


In [ ]:
class CNNClassique(nn.Module):
    def __init__(self, dropout=0.3):
        super().__init__()
        self.bloc1 = nn.Sequential(nn.Conv2d(1,16,3,padding=1),nn.ReLU(),nn.MaxPool2d(2))
        self.bloc2 = nn.Sequential(nn.Conv2d(16,32,3,padding=1),nn.ReLU(),nn.MaxPool2d(2))
        self.bloc3 = nn.Sequential(nn.Conv2d(32,64,3,padding=1),nn.ReLU(),nn.MaxPool2d(2))
        self.reduction = nn.AdaptiveAvgPool2d((4,4))
        self.classificateur = nn.Sequential(nn.Flatten(),nn.Linear(64*4*4,64),nn.ReLU(),nn.Dropout(dropout),nn.Linear(64,2))
    def forward(self,x):
        x=self.bloc3(self.bloc2(self.bloc1(x)))
        return self.classificateur(self.reduction(x))

class AttentionCanaux(nn.Module):
    def __init__(self, canaux, reduction=16):
        super().__init__()
        dim=max(1,canaux//reduction)
        self.reseau_partage=nn.Sequential(
            nn.Conv2d(canaux,dim,1,bias=False),nn.ReLU(),
            nn.Conv2d(dim,canaux,1,bias=False))
    def forward(self,x):
        moyenne=x.mean(dim=(2,3),keepdim=True)
        maximum=x.amax(dim=(2,3),keepdim=True)
        return x*torch.sigmoid(self.reseau_partage(moyenne)+self.reseau_partage(maximum))

class AttentionSpatiale(nn.Module):
    def __init__(self):
        super().__init__()
        self.convolution=nn.Conv2d(2,1,7,padding=3,bias=False)
    def forward(self,x):
        resume=torch.cat([x.mean(dim=1,keepdim=True),x.amax(dim=1,keepdim=True)],dim=1)
        return x*torch.sigmoid(self.convolution(resume))

class CBAM(nn.Module):
    def __init__(self,canaux=64,reduction=16):
        super().__init__()
        self.attention_canaux=AttentionCanaux(canaux,reduction)
        self.attention_spatiale=AttentionSpatiale()
    def forward(self,x):
        return self.attention_spatiale(self.attention_canaux(x))

class CNNAvecCBAM(CNNClassique):
    def __init__(self,dropout=0.3):
        super().__init__(dropout)
        self.cbam=CBAM(64,16)
    def forward(self,x):
        x=self.bloc3(self.bloc2(self.bloc1(x)))
        return self.classificateur(self.reduction(self.cbam(x)))

with torch.random.fork_rng(devices=[]):
    modele_verification = CNNAvecCBAM().eval()
    x, y = next(iter(DataLoader(TRAIN, batch_size=LOT, shuffle=False, num_workers=0)))
    with torch.no_grad(): scores = modele_verification(x)
    assert scores.shape == (LOT, 2) and torch.isfinite(scores).all()
    assert sum(p.numel() for p in modele_verification.parameters()) == 89636
del modele_verification
print("Architecture vérifiée : 89 636 paramètres, deux scores par image.")


## Fonctions de reprise et de mesure
Les moyennes utilisent toutes les graines. L’écart-type décrit leur dispersion ; il ne constitue pas un intervalle de confiance.

In [ ]:
def ecrire_json(obj, chemin):
    tmp = chemin.with_suffix(".tmp")
    tmp.write_text(json.dumps(obj, ensure_ascii=False, indent=2), encoding="utf-8")
    tmp.replace(chemin)

def sauver_torch(obj, chemin):
    tmp = chemin.with_suffix(".tmp")
    torch.save(obj, tmp)
    tmp.replace(chemin)

def construire_protocole(architecture, graine):
    protocole = {
        "architecture": architecture, "graine": graine,
        "lot": LOT, "max_epoques": MAX_EPOQUES, "patience": PATIENCE,
        "critere": "val_f1_macro", "amelioration": "strictement superieure",
        "grille": [list(x) for x in GRILLE], "optimiseur": "Adam", "weight_decay": 0.0,
        "pretraitement": "L; ImageOps.pad 224 bilineaire noir centre; /255; aucune augmentation",
        "classes": CLASSES, "poids_classes": POIDS.tolist(),
        "sha256_csv": {s: hashlib.sha256((REPARTITION / f"{s}.csv").read_bytes()).hexdigest() for s in ["train", "val"]},
        "versions": {"python": sys.version, "torch": str(torch.__version__), "numpy": np.__version__, "pandas": pd.__version__, "pillow": PIL.__version__, "sklearn": version_sklearn},
        "threads_torch": torch.get_num_threads(),
        "limite": "Stabilité sur répartition fixe; test déjà consulté; graine 42 utilisée pour sélectionner les hyperparamètres."
    }
    return protocole

def une_epoque(modele, chargeur, critere, optimiseur=None):
    apprentissage = optimiseur is not None
    modele.train(apprentissage)
    pertes, poids_total = 0.0, 0.0
    reels, predits = [], []
    for images, etiquettes in tqdm(chargeur, desc="Apprentissage" if apprentissage else "Validation", leave=False):
        images, etiquettes = images.to(APPAREIL), etiquettes.to(APPAREIL)
        if apprentissage: optimiseur.zero_grad(set_to_none=True)
        with torch.set_grad_enabled(apprentissage):
            scores = modele(images)
            perte = critere(scores, etiquettes)
            if not torch.isfinite(perte).item(): raise RuntimeError("Perte non finie : arrêter et examiner cette configuration.")
            if apprentissage:
                perte.backward()
                optimiseur.step()
        poids = critere.weight[etiquettes].sum().item()
        pertes += perte.item()*poids
        poids_total += poids
        reels.extend(etiquettes.cpu().tolist())
        predits.extend(scores.detach().argmax(1).cpu().tolist())
    cm = confusion_matrix(reels, predits, labels=[0,1])
    tn,fp,fn,tp = cm.ravel()
    mesures = {
        "perte": float(pertes/poids_total), "exactitude": float(accuracy_score(reels,predits)),
        "precision_pneumonie": float(precision_score(reels,predits,zero_division=0)),
        "rappel_pneumonie": float(recall_score(reels,predits,zero_division=0)),
        "specificite_NORMAL": float(tn/(tn+fp)),
        "f1_pneumonie": float(f1_score(reels,predits,zero_division=0)),
        "f1_macro": float(f1_score(reels,predits,labels=[0,1],average="macro",zero_division=0))
    }
    return mesures, cm.tolist()

def lancer_configuration(indice, lr, dropout):
    dossier = SORTIE / f"configuration_{indice:02d}"
    dossier.mkdir(exist_ok=True)
    configuration = {"indice": indice, "lr": lr, "dropout": dropout}
    torch.manual_seed(GRAINE)
    modele = CLASSE_MODELE(dropout).to(APPAREIL)
    optimiseur = torch.optim.Adam(modele.parameters(), lr=lr)
    critere = nn.CrossEntropyLoss(weight=POIDS.to(APPAREIL))
    generateur = torch.Generator().manual_seed(GRAINE)
    chargeur_train = DataLoader(TRAIN,batch_size=LOT,shuffle=True,generator=generateur,num_workers=0)
    chargeur_val = DataLoader(VAL,batch_size=LOT,shuffle=False,num_workers=0)
    # Graine d'entraînement commune, indépendante du nombre de paramètres initialisés.
    torch.manual_seed(GRAINE)
    historique = []
    def sauvegarder(epoque, matrice=None):
        sauver_torch({"epoque":epoque,"modele":modele.state_dict(),"optimiseur":optimiseur.state_dict(),
                      "historique":historique,"configuration":configuration,"protocole":PROTOCOLE,
                      "etat_torch":torch.get_rng_state(),"etat_generateur":generateur.get_state(),
                      "matrice_validation":matrice}, dossier/f"epoque_{epoque:03d}.pt")
    disponibles = sorted(dossier.glob("epoque_[0-9][0-9][0-9].pt"))
    if disponibles:
        etat = torch.load(disponibles[-1], map_location="cpu", weights_only=True)
        assert etat["configuration"] == configuration and etat["protocole"] == PROTOCOLE
        modele.load_state_dict(etat["modele"],strict=True)
        optimiseur.load_state_dict(etat["optimiseur"])
        historique = etat["historique"]
        assert etat["epoque"] == len(historique)
        torch.set_rng_state(etat["etat_torch"])
        generateur.set_state(etat["etat_generateur"])
        print("Reprise après l'époque", etat["epoque"])
    else:
        sauvegarder(0)
    while len(historique) < MAX_EPOQUES:
        if historique:
            meilleur = max(historique,key=lambda r:r["val_f1_macro"])
            if len(historique)-meilleur["epoque"] >= PATIENCE: break
        epoque = len(historique)+1
        debut = time.perf_counter()
        tr,_ = une_epoque(modele,chargeur_train,critere,optimiseur)
        va,cm = une_epoque(modele,chargeur_val,critere)
        ligne = {"epoque":epoque,"duree_secondes":float(time.perf_counter()-debut)}
        ligne.update({f"train_{k}":v for k,v in tr.items()})
        ligne.update({f"val_{k}":v for k,v in va.items()})
        historique.append(ligne)
        sauvegarder(epoque,cm)
        pd.DataFrame(historique).to_csv(dossier/"historique.csv",index=False)
        print(f"Époque {epoque:02d} | F1 macro val {va['f1_macro']:.4f} | rappel {va['rappel_pneumonie']:.4f} | spécificité {va['specificite_NORMAL']:.4f}")
    meilleur = max(historique,key=lambda r:r["val_f1_macro"])
    source = dossier/f"epoque_{meilleur['epoque']:03d}.pt"
    tmp = dossier/"meilleur.tmp"
    shutil.copy2(source,tmp)
    tmp.replace(dossier/"meilleur.pt")
    pd.DataFrame(historique).to_csv(dossier/"historique.csv",index=False)
    resume = {**configuration,"epoque_retenue":meilleur["epoque"],"epoque_finale":len(historique),
              "arret": "patience" if len(historique)-meilleur["epoque"]>=PATIENCE else "maximum_epoques",
              "duree_minutes":sum(r["duree_secondes"] for r in historique)/60,
              "sauvegarde":str(dossier/"meilleur.pt"),
              **{k:v for k,v in meilleur.items() if k.startswith("val_")}}
    ecrire_json(resume,dossier/"resume.json")
    return resume

print("Fonctions et protocole prêts. Aucun entraînement lancé jusqu'ici.")


In [ ]:
ARCHITECTURES = {
    "CNN_classique": (CNNClassique, "CNNClassique_89026_v1", "cnn_classique_grille_v1"),
    "CNN_CBAM": (CNNAvecCBAM, "CNNAvecCBAM_89636_apres_bloc3_r16_k7_v1", "cnn_cbam_grille_v1"),
}
# Lire les protocoles d'origine pour vérifier aussi l'identifiant exact d'architecture.
BASES = []
for nom, (classe, architecture, ancien_dossier) in ARCHITECTURES.items():
    dossier = PROJET / "optimisation" / ancien_dossier
    protocole_initial = json.loads((dossier / "protocole.json").read_text(encoding="utf-8"))
    attendu = construire_protocole(architecture, 42)
    # L'identifiant du CNN classique est repris du notebook d'origine ci-dessous.
    for cle, valeur in attendu.items():
        if cle in {"grille", "limite"}: continue
        if protocole_initial.get(cle) != valeur:
            raise RuntimeError(f"{nom} : protocole différent pour {cle}. Vérifier avant de poursuivre.")
    chemin = dossier / "configuration_02" / "meilleur.pt"
    etat = torch.load(chemin, map_location="cpu", weights_only=True)
    if etat["protocole"] != protocole_initial or etat["configuration"] != {"indice": 2, "lr": 0.001, "dropout": 0.5}:
        raise RuntimeError(f"Sauvegarde incompatible : {chemin}")
    modele = classe(dropout=0.5)
    modele.load_state_dict(etat["modele"], strict=True)
    meilleur = max(etat["historique"], key=lambda r:r["val_f1_macro"])
    if meilleur["epoque"] != etat["epoque"]:
        raise RuntimeError("La sauvegarde fournie n'est pas celle de la meilleure époque.")
    BASES.append({"modele": nom, "graine": 42, "lr": 0.001, "dropout": 0.5,
                  "epoque_retenue": etat["epoque"], "sauvegarde": str(chemin),
                  **{k:v for k,v in meilleur.items() if k.startswith("val_")}})
    del modele, etat
RACINE.mkdir(parents=True, exist_ok=True)
display(pd.DataFrame(BASES)[["modele", "graine", "epoque_retenue", "val_f1_macro"]])
print("Sauvegardes de la graine 42 vérifiées. Aucun entraînement lancé.")


## Lancer les quatre entraînements
Laisser cette cellule terminer. Le tableau est enregistré après chaque entraînement. Pour interrompre, utiliser le bouton d’arrêt puis reprendre ultérieurement dans ce même notebook.

In [ ]:
resumes = list(BASES)
for nom, (CLASSE_MODELE, architecture, _) in ARCHITECTURES.items():
    for GRAINE in [123, 2026]:
        SORTIE = RACINE / nom / f"graine_{GRAINE}"
        SORTIE.mkdir(parents=True, exist_ok=True)
        PROTOCOLE = construire_protocole(architecture, GRAINE)
        fichier = SORTIE / "protocole.json"
        if fichier.exists():
            if json.loads(fichier.read_text(encoding="utf-8")) != PROTOCOLE:
                raise RuntimeError(f"Protocole de reprise différent : {SORTIE}")
        else:
            ecrire_json(PROTOCOLE, fichier)
        print(f"\n===== {nom} | graine {GRAINE} =====")
        resume = lancer_configuration(2, 0.001, 0.5)
        resumes.append({"modele": nom, "graine": GRAINE, **resume})
        bilan = pd.DataFrame(resumes)
        bilan.to_csv(RACINE / "resultats_multigraines.csv", index=False, encoding="utf-8-sig")
        display(bilan[["modele", "graine", "epoque_retenue", "val_f1_macro", "val_rappel_pneumonie", "val_specificite_NORMAL"]].round(5))

metriques = ["val_f1_macro", "val_exactitude", "val_rappel_pneumonie", "val_specificite_NORMAL"]
def resumer(tableau):
    sortie = tableau.groupby("modele")[metriques].agg(["mean", "std", "min", "max"])
    sortie.columns = [f"{mesure}_{stat}" for mesure, stat in sortie.columns]
    return sortie.reset_index()
synthese = resumer(bilan)
synthese.to_csv(RACINE / "synthese_multigraines.csv", index=False, encoding="utf-8-sig")
resumer(bilan[bilan["graine"] != 42]).to_csv(RACINE / "synthese_nouvelles_graines.csv", index=False, encoding="utf-8-sig")
comparaison = bilan.pivot(index="graine", columns="modele", values="val_f1_macro")
comparaison["ecart_CBAM_moins_classique"] = comparaison["CNN_CBAM"] - comparaison["CNN_classique"]
comparaison.to_csv(RACINE / "ecarts_par_graine.csv", encoding="utf-8-sig")
display(synthese.round(5))
display(comparaison.round(5))
print("Terminé. Fichier à transmettre :", RACINE / "resultats_multigraines.csv")
print("Aucun test exécuté. Aucune graine sélectionnée comme gagnante.")
